# Local LLM Pull Request Review

Review a local Git diff with Ollama and save the findings as Markdown. The default endpoint is `localhost`, so the diff stays on your machine.

## Setup

1. Start Ollama and download a model, for example: `ollama pull gemma3:4b`.
2. Install dependencies: `uv sync`.
3. Register the kernel: `uv run python -m ipykernel install --user --name local-pr-llm-review --display-name \"Local PR LLM Review\"`.
4. Copy `.env.example` to `.env` to change the model or endpoint.
5. Start Jupyter with `uv run jupyter notebook`, select **Local PR LLM Review**, and run the cells in order.

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

PROJECT_DIR = Path(r"C:\path-to-your-project")
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'gemma3:4b')
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY', 'ollama')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)
print(f'Project: {PROJECT_DIR}')
print(f'Local LLM endpoint: {OLLAMA_BASE_URL}')
print(f'Model: {OLLAMA_MODEL}')

Project: C:\Users\Enriek\Projects\github.com\Tools
Local LLM endpoint: http://localhost:11434/v1
Model: gemma3:4b


In [2]:
# Confirm that Ollama is reachable and the selected model is installed.
models = client.models.list()
available_models = [model.id for model in models.data]
print('Available models:', available_models)
if OLLAMA_MODEL not in available_models:
    print(f"Warning: {OLLAMA_MODEL!r} was not listed. Run: ollama pull {OLLAMA_MODEL}")
else:
    print('Ollama is ready.')

Available models: ['deepseek-r1:1.5b', 'llama3.2:latest', 'gemma3:4b', 'deepseek-v3.1:671b-cloud', 'gemma3:270m', 'asuna-bot:latest', 'llama3.2:3b']
Ollama is ready.


## Choose a diff source

Set `DIFF_FILE` to review an existing `.diff` file. Otherwise, use `working_tree` to review local staged and unstaged changes against `HEAD`, or use `refs` to compare two local Git refs. The notebook does not run `git fetch`.

In [ ]:
# Option A: point to an existing diff file. Example: Path(r'C:/temp/my-change.diff')
DIFF_FILE: Path | None = None

# Option B: generate a diff from a local Git repository.
REPO_DIR = PROJECT_DIR  # Example: Path(r'C:\path-to-your-project')

# Used only when DIFF_SOURCE = 'refs'. Use origin/main when local main contains the commits being reviewed.
BASE_REF = 'origin/main'
HEAD_REF = 'HEAD'

# Deliberate safety limit for larger reviews.
MAX_DIFF_CHARS = 60_000

In [ ]:
def run_git_diff(repo_dir: Path, *refs: str) -> str:
    command = ['git', '-C', str(repo_dir), 'diff', '--minimal', '--find-renames']
    command.extend(refs)
    command.extend(['--',
         ':(exclude)node_modules/**', ':(exclude)dist/**', ':(exclude)vendor/**',
         ':(exclude)**/*.lock', ':(exclude)**/*.min.js', ':(exclude)**/*.map',
         ':(exclude)**/*.png', ':(exclude)**/*.jpg', ':(exclude)**/*.jpeg', ':(exclude)**/*.gif'])

    result = subprocess.run(
        command,
        capture_output=True, text=True, check=True, encoding='utf-8', errors='replace',
    )
    return result.stdout

def redact_diff(text: str) -> str:
    """Basic defence-in-depth redaction; extend for your organisation's patterns."""
    replacements = [
        (r'https?://[^\s\"\')]+', '<URL>'),
        (r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '<EMAIL>'),
        (r'(?i)(Bearer\s+)[A-Za-z0-9._=-]+', r'\1<TOKEN>'),
        (r'eyJ[A-Za-z0-9._=-]+', '<JWT>'),
        (r'(?i)(api[_-]?key\s*[:=]\s*)[\"\']?[^\"\',\s]+', r'\1<API_KEY>'),
    ]
    for pattern, replacement in replacements:
        text = re.sub(pattern, replacement, text)
    return text

if DIFF_FILE:
    raw_diff = DIFF_FILE.read_text(encoding='utf-8')
else:
    raw_diff = run_git_diff(REPO_DIR, f'{BASE_REF}...{HEAD_REF}')
    raise ValueError("DIFF_SOURCE must be 'working_tree' or 'refs'.")
redacted_diff = redact_diff(raw_diff)

if not redacted_diff.strip():
    raise ValueError('The diff is empty. For local changes, check git status; untracked files are not included. For branch comparison, set DIFF_SOURCE to refs and check BASE_REF / HEAD_REF.')
if len(redacted_diff) > MAX_DIFF_CHARS:
    raise ValueError(f'The diff has {len(redacted_diff):,} characters, exceeding the {MAX_DIFF_CHARS:,} character limit. Reduce the scope or raise the limit deliberately.')

print(f'Diff ready: {len(redacted_diff):,} characters')
display(Markdown(f'```diff\n{redacted_diff[:4_000]}\n```'))

Diff ready: 12,439 characters


```diff
diff --git a/local-pr-llm-review/.env.example b/local-pr-llm-review/.env.example
new file mode 100644
index 0000000..d7c83ac
--- /dev/null
+++ b/local-pr-llm-review/.env.example
@@ -0,0 +1,5 @@
+# Ollama exposes an OpenAI-compatible API locally.
+OLLAMA_BASE_URL=<URL>
+OLLAMA_MODEL=gemma3:4b
+# Ollama does not require an API key, but the OpenAI client requires a value.
+OLLAMA_API_KEY=<API_KEY>
diff --git a/local-pr-llm-review/.gitignore b/local-pr-llm-review/.gitignore
new file mode 100644
index 0000000..c1b4ad3
--- /dev/null
+++ b/local-pr-llm-review/.gitignore
@@ -0,0 +1,6 @@
+.env
+.venv/
+.uv-cache/
+.uv-python/
+__pycache__/
+outputs/
diff --git a/local-pr-llm-review/local_llm_pr_review.ipynb b/local-pr-llm-review/local_llm_pr_review.ipynb
new file mode 100644
index 0000000..8320e3d
--- /dev/null
+++ b/local-pr-llm-review/local_llm_pr_review.ipynb
@@ -0,0 +1,258 @@
+{
+ "cells": [
+  {
+   "cell_type": "markdown",
+   "metadata": {},
+   "source": [
+    "# Local LLM Pull Request Review\n",
+    "\n",
+    "Review a local Git diff with Ollama and save the findings as Markdown. The default endpoint is `localhost`, so the diff stays on your machine.\n",
+    "\n",
+    "## Setup\n",
+    "\n",
+    "1. Start Ollama and download a model, for example: `ollama pull gemma3:4b`.\n",
+    "2. Install dependencies: `uv sync`.\n",
+    "3. Register the kernel: `uv run python -m ipykernel install --user --name local-pr-llm-review --display-name \\\"Local PR LLM Review\\\"`.\n",
+    "4. Copy `.env.example` to `.env` to change the model or endpoint.\n",
+    "5. Start Jupyter with `uv run jupyter notebook`, select **Local PR LLM Review**, and run the cells in order."
+   ]
+  },
+  {
+   "cell_type": "code",
+   "execution_count": null,
+   "metadata": {},
+   "outputs": [
+    {
+     "name": "stdout",
+     "output_type": "stream",
+     "text": [
+      "Project: c:\\Users\\Enriek\\Projects\\github.com\\Tools\\local-pr-llm-review\n",
+      "Local LLM endpoint: <URL>",
+      "Model: gemma3:4b\n"
+     ]
+    }
+   ],
+   "source": [
+    "from __future__ import annotations\n",
+    "\n",
+    "import os\n",
+    "import re\n",
+    "import subprocess\n",
+    "from datetime import datetime\n",
+    "from pathlib import Path\n",
+    "\n",
+    "from dotenv import load_dotenv\n",
+    "from IPython.display import Markdown, display\n",
+    "from openai import OpenAI\n",
+    "\n",
+    "load_dotenv()\n",
+    "\n",
+    "PROJECT_DIR = Path(r\"C:\\Users\\Enriek\\Projects\\github.com\\Tools\")\n",
+    "OUTPUT_DIR = PROJECT_DIR / 'outputs'\n",
+    "OUTPUT_DIR.mkdir(exist_ok=True)\n",
+    "\n",
+    "OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', '<URL>')\n",
+    "OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'gemma3:4b')\n",
+    "OLLAMA_API_KEY = <API_KEY>'OLLAMA_API_KEY', 'ollama')\n",
+    "\n",
+    "client = OpenAI(base_url=OLLAMA_BASE_URL, api_key=<API_KEY>",
+    "print(f'Project: {PROJECT_DIR}')\n",
+    "print(f'Local LLM endpoint: {OLLAMA_BASE_URL}')\n",
+    "print(f'Model: {OLLAMA_MODEL}')"
+   ]
+  },
+  {
+   "cell_type": "code",
+   "execution_count": 3,
+   "metadata": {},
+   "outputs": [
+    {
+     "name": "stdout",
+     "output_type": "stream",
+     "text": [
+      "Available models: ['deepseek-r1:1.5b', 'llama3.2:latest', 'gemma3:4b', 'deepseek-v3.1:671b-cloud', 'gemma3:270m', 'asuna-bot:latest', 'llama3.2:3b']\n",
+      "Ollama is ready.\n"
+     ]
+    }
+   ],
+   "source": [
+    "# Confirm that Ollama is reachable and the selected model is installed.\n",
+    "models = client.models.list()\n",
+    "available_models = [model.id for model in models.data]\n",
+    "print('Available models:', available_models)\n",
+    "if OLLAMA_MODEL not in available_models:\n",
+    "    print(f\"Warning: {OLLAMA_MODEL!r} was not listed. Run: ollama pull {OLLAMA_MODEL}\")\n",
+    "else:\n",
+    "    print('Ollama is ready.')"
+   ]
+  },
+  {
+   "cell_type": "markdown",
+   "metadata": {},
+   "source": [
+    "## Ch
```

In [5]:
SYSTEM_PROMPT = """You are a senior code reviewer. Review only the supplied diff.
Find concrete correctness, security, reliability, or maintainability problems introduced by this change.
Do not invent surrounding code. Do not praise or summarize unless needed.
For every finding, write: severity (P0-P3), file and line, why it is a problem, and a concise fix.
If there are no actionable findings, respond exactly: No actionable findings."""

USER_PROMPT = f"""Review this redacted Git diff:

```diff
{redacted_diff}
```"""

In [6]:
response = client.chat.completions.create(
    model=OLLAMA_MODEL,
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_PROMPT},
    ],
)

review = response.choices[0].message.content or 'No review content returned.'
display(Markdown(review))

Okay, I've reviewed the provided code and documentation. Here's a breakdown of my assessment, focusing on potential improvements and considerations:

**Overall Assessment:**

This is a well-structured and thoughtfully designed system for using an LLM to review Git diffs locally. The modular design with functions like `redact_diff` and the use of prompts are excellent choices.  The inclusion of error handling (empty diff, character limit) adds robustness. However, there's room for refinement in several areas, particularly around prompt engineering, security considerations, and logging/reporting.

**Detailed Review & Recommendations:**

1. **`redact_diff` Function:**
   - **Good:** The regular expressions are a solid starting point for common redaction targets (URLs, emails, API keys, JWTs).  The case-insensitive matching (`(?i)`) is helpful.
   - **Potential Improvements:**
     * **More Comprehensive Regexes:** Consider adding more regex patterns to cover other potential sensitive data formats you might encounter (e.g., base64 encoded strings, serialized objects).
     * **Specificity:** Some of the regexes are broad.  For example, `[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}` is a standard email regex but could be refined to reduce false positives if you know more about your codebase's email usage.
     * **Order of Redactions:** The order in which the replacements are applied *can* matter.  If one replacement creates a pattern that's then matched by another, it can lead to unexpected behavior. Consider ordering them based on likelihood or complexity.

2. **Prompt Engineering (SYSTEM_PROMPT & USER_PROMPT):**
   - **Good:** The `SYSTEM_PROMPT` is well-defined and sets clear expectations for the LLM's role.
   - **Critical Improvement:  USER_PROMPT:** This is currently just a verbatim copy of the redacted diff. *This is the single most important area to improve.* The quality of the review will be entirely dependent on how effectively you frame the prompt. Here are some suggestions:
     * **Contextual Information:** Add context about the change being reviewed (e.g., "You are reviewing a Git diff for a feature that adds user authentication.").  This helps the LLM understand the purpose and potential impact of the changes.
     * **Specific Instructions:** Be more explicit about what you want the LLM to look for: “Identify any security vulnerabilities, code smells, or areas where the logic could be improved.”  You can even provide examples of the types of issues you're interested in.
     * **Output Format:** Request a specific output format (e.g., "Provide your findings as a numbered list with severity, file/line number, and a concise description of the issue.").

3. **Error Handling:**
   - **Good:** The error handling for empty diffs and character limits is sensible.
   - **Considerations:**
     * **More Granular Errors:**  The `run_git_diff` function could potentially raise more specific Git errors (e.g., if the branch doesn't exist). Catching these and providing more informative error messages would be beneficial.

4. **Security Considerations:**
   - **Important:** The code currently performs redaction *before* sending the diff to the LLM. This is a good start, but it’s not sufficient.  You need to consider:
     * **Prompt Injection:** Even with redaction, there's still a risk of prompt injection attacks where malicious input could trick the LLM into ignoring your instructions or revealing sensitive information. Carefully sanitize and validate any user-provided inputs (though in this case, it’s just the diff).
     * **LLM Security:** Be aware that the LLM itself might have vulnerabilities.  Use a reputable LLM provider with strong security practices.

5. **Logging & Reporting:**
   - **Good:** Saving the review to an MD file is a good practice.
   - **Improvements:**
     * **Detailed Logging:** Add more logging throughout the code (e.g., timestamps, LLM response times, any errors encountered). This will help with debugging and monitoring.
     * **Structured Reporting:**  Consider using a structured format for the report (e.g., JSON) to make it easier to process programmatically.

6. **`DIFF_SOURCE` Handling:**
   - **Good:** The logic for determining the `DIFF_SOURCE` is clear.
   - **Recommendation:** Add a default value for `DIFF_SOURCE` if none is provided, and log a warning message indicating that it's being set to the default.

7. **Dependencies:**
    -  The `pyproject.toml` file correctly lists dependencies. Ensure these are properly installed using `uv sync`.


**Revised Code Snippet (Illustrative - Focusing on USER_PROMPT):**

```python
USER_PROMPT = f"""Review this redacted Git diff:

```diff
{redacted_diff}
```

You are a senior code reviewer.  Find concrete correctness, security, reliability, or maintainability problems introduced by this change. The diff is for a feature that adds user authentication. Do not invent surrounding code. Do not praise or summarize unless needed. For every finding, write: severity (P0-P3), file and line, why it is a problem, and a concise fix.  Focus on potential vulnerabilities related to authentication.
"""
```

**In summary:** This is a solid foundation for an automated code review tool. By focusing on prompt engineering, security best practices, and robust logging, you can significantly improve its effectiveness and reliability. Remember that the LLM's performance will be heavily influenced by the quality of your prompts – invest time in crafting them carefully!


In [ ]:
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
report_path = OUTPUT_DIR / f'local-llm-review-{timestamp}.md'
report_path.write_text(
    f'# Local LLM review\n\nModel: `{OLLAMA_MODEL}`\n\n## Findings\n\n{review}\n',
    encoding='utf-8',
)
print(f'Review saved to: {report_path}')